# Multi-Order Block vs TWAP (Teaching Notebook)

This notebook follows the same teaching style as `LOB_Simulation_Live_Demos.ipynb` and focuses on the execution pipeline for:
- `simulate_multi_order_block(...)`
- `simulate_multi_order_twap(...)`

Sign convention:
- `qty > 0` = BUY
- `qty < 0` = SELL

Reproducibility:
- We use fixed deterministic baseline inputs (including `seed`) and compare Block vs TWAP side-by-side in each section.


In [2]:
# 1) Add project folder to import path (run once per kernel)
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent   # if notebook is in presentation_assets/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:

from pprint import pprint
import inspect

from Execution_Methods.Block_Trade import simulate_multi_order_block
from Execution_Methods.TWAP import simulate_multi_order_twap
import LOB_Simulation

# Helper for compact side-by-side table printing.
def print_side_by_side_rows(rows, col1="Metric", col2="Block", col3="TWAP"):
    print(f"{col1:<24} | {col2:<24} | {col3:<24}")
    print("-" * 80)
    for k, v1, v2 in rows:
        print(f"{str(k):<24} | {str(v1):<24} | {str(v2):<24}")


In [4]:
# Baseline deterministic inputs shared by all sections.
# Same structure and parameter style as previous notebook demos.

parent_orders = {5: -10000, 20: -5000, 35: 4000}

ob_params = {
    "reference_price": 28.13,
    "tick_size": 0.01,
    "n_levels": 10,
    "seed": 42,
}

sim_params = {
    "bg_lam": 3.5,
    "bg_p_buy": 0.5,
    "rho": 1.0,
    "noise": 0.10,
    "perm_eta_ticks": 0.0,
    "perm_gamma": 1.0,
    "perm_scale": 7500,
    "main_exec_kwargs": {
        "impact_eta_ticks": 0.2,
        "impact_gamma": 1.6,
        "impact_scale": 7500,
        "impact_use_cum": True,
    },
    "bg_exec_kwargs": {
        "impact_eta_ticks": 0.0,
        "impact_gamma": 1.5,
        "impact_scale": 7500,
        "impact_use_cum": True,
    },
    "order": "bg_first",
}

time_horizon = 20

print("Baseline inputs set.")
print("parent_orders:")
pprint(parent_orders)
print()
print("ob_params:")
pprint(ob_params)
print()
print("sim_params (top-level keys):", list(sim_params.keys()))
print("time_horizon:", time_horizon)


Baseline inputs set.
parent_orders:
{5: -10000, 20: -5000, 35: 4000}

ob_params:
{'n_levels': 10, 'reference_price': 28.13, 'seed': 42, 'tick_size': 0.01}

sim_params (top-level keys): ['bg_lam', 'bg_p_buy', 'rho', 'noise', 'perm_eta_ticks', 'perm_gamma', 'perm_scale', 'main_exec_kwargs', 'bg_exec_kwargs', 'order']
time_horizon: 20


In [5]:
# Section 1: Input Contract
# This section explains what each function expects and prints concrete values used in this notebook.

print("Function signatures:")
print("- Block:", inspect.signature(simulate_multi_order_block))
print("- TWAP :", inspect.signature(simulate_multi_order_twap))

rows = [
    ("parent_orders", "required", "required"),
    ("ob_params", "required", "required"),
    ("sim_params", "required", "required"),
    ("time_horizon", "not used", "required"),
]
print()
print("Input contract (side-by-side):")
print_side_by_side_rows(rows, col1="Input", col2="Block", col3="TWAP")

print()
print("Concrete deterministic examples from this notebook:")
print("parent_orders =", parent_orders)
print("ob_params =", ob_params)
print("sim_params =", {k: sim_params[k] for k in ["bg_lam", "bg_p_buy", "rho", "noise", "order"]})
print("time_horizon =", time_horizon)

print()
print("Teaching notes:")
print("- Block executes each parent order directly at its arrival time.")
print("- TWAP slices each parent order across `time_horizon` steps starting from arrival.")


Function signatures:
- Block: (parent_orders, ob_params, sim_params)
- TWAP : (parent_orders, time_horizon, ob_params, sim_params)

Input contract (side-by-side):
Input                    | Block                    | TWAP                    
--------------------------------------------------------------------------------
parent_orders            | required                 | required                
ob_params                | required                 | required                
sim_params               | required                 | required                
time_horizon             | not used                 | required                

Concrete deterministic examples from this notebook:
parent_orders = {5: -10000, 20: -5000, 35: 4000}
ob_params = {'reference_price': 28.13, 'tick_size': 0.01, 'n_levels': 10, 'seed': 42}
sim_params = {'bg_lam': 3.5, 'bg_p_buy': 0.5, 'rho': 1.0, 'noise': 0.1, 'order': 'bg_first'}
time_horizon = 20

Teaching notes:
- Block executes each parent order directly a

In [6]:
# Section 2: Schedule Construction
# Side-by-side schedule preview: Block direct schedule vs TWAP sliced + netted schedule.

# Block schedule is exactly the parent order map.
block_schedule = dict(parent_orders)

# TWAP schedule: slice each parent order over `time_horizon` and net overlaps by time step.
twap_schedule = {}
for arrival_t, qty in parent_orders.items():
    sign = 1 if qty > 0 else -1
    abs_qty = abs(qty)
    slice_size = abs_qty // time_horizon
    remainder = abs_qty % time_horizon

    for i in range(time_horizon):
        t = arrival_t + i
        chunk = slice_size + (remainder if i == time_horizon - 1 else 0)
        twap_schedule[t] = twap_schedule.get(t, 0) + sign * chunk

# Show a compact combined view over all active timesteps.
all_t = sorted(set(block_schedule.keys()) | set(twap_schedule.keys()))
rows = [(t, block_schedule.get(t, 0), twap_schedule.get(t, 0)) for t in all_t]

print("Time-schedule comparison (qty at each step):")
print_side_by_side_rows(rows, col1="t", col2="Block qty", col3="TWAP net qty")

print()
print("Teaching notes:")
print("- Block has activity only at original arrival timestamps.")
print("- TWAP spreads each order over many steps, so per-step footprint is smaller.")
print("- Overlapping TWAP slices are netted into one signed quantity per step.")


Time-schedule comparison (qty at each step):
t                        | Block qty                | TWAP net qty            
--------------------------------------------------------------------------------
5                        | -10000                   | -500                    
6                        | 0                        | -500                    
7                        | 0                        | -500                    
8                        | 0                        | -500                    
9                        | 0                        | -500                    
10                       | 0                        | -500                    
11                       | 0                        | -500                    
12                       | 0                        | -500                    
13                       | 0                        | -500                    
14                       | 0                        | -500                    
15   

In [7]:
# Section 3: LOB Initialization
# We initialize one book for Block preview and one for TWAP preview using the same parameters.
# At initialization, they should match because no execution method has been applied yet.

first_parent_abs_qty = abs(next(iter(parent_orders.values())))

def init_book_for_preview():
    ob = LOB_Simulation.OrderBook(**ob_params)
    ob.order_book_levels(
        execution_qty=first_parent_abs_qty,
        noise=0.1,
        mult_low=5.0,
        mult_high=10.0,
    )
    return ob

ob_block_init = init_book_for_preview()
ob_twap_init = init_book_for_preview()

block_best_bid = ob_block_init.bids[0] if ob_block_init.bids else (None, None)
block_best_ask = ob_block_init.asks[0] if ob_block_init.asks else (None, None)
twap_best_bid = ob_twap_init.bids[0] if ob_twap_init.bids else (None, None)
twap_best_ask = ob_twap_init.asks[0] if ob_twap_init.asks else (None, None)

rows = [
    ("best_bid", block_best_bid, twap_best_bid),
    ("best_ask", block_best_ask, twap_best_ask),
    ("top3_bids", ob_block_init.bids[:3], ob_twap_init.bids[:3]),
    ("top3_asks", ob_block_init.asks[:3], ob_twap_init.asks[:3]),
]

print("Initialized LOB preview (side-by-side):")
print_side_by_side_rows(rows)

print()
print("Teaching notes:")
print("- Both strategies start from equivalent book states with shared deterministic inputs.")
print("- Differences emerge later from scheduling/execution logic, not from initialization.")


Initialized LOB preview (side-by-side):
Metric                   | Block                    | TWAP                    
--------------------------------------------------------------------------------
best_bid                 | (28.1, 1287)             | (28.1, 1287)            
best_ask                 | (28.16, 1223)            | (28.16, 1223)           
top3_bids                | [(28.1, 1287), (28.09, 8874), (28.080000000000002, 13989)] | [(28.1, 1287), (28.09, 8874), (28.080000000000002, 13989)]
top3_asks                | [(28.16, 1223), (28.17, 8334), (28.18, 13019)] | [(28.16, 1223), (28.17, 8334), (28.18, 13019)]

Teaching notes:
- Both strategies start from equivalent book states with shared deterministic inputs.
- Differences emerge later from scheduling/execution logic, not from initialization.


In [8]:
# Section 4: Simulation Run
# Run both strategies on the same inputs and store full outputs for Section 5.

block_result = simulate_multi_order_block(parent_orders, ob_params, sim_params)
twap_result = simulate_multi_order_twap(parent_orders, time_horizon, ob_params, sim_params)

key_metrics = ["strategy", "net_requested_qty", "net_filled_qty", "arrival_vwap", "global_vwap", "slippage"]
rows = [(k, block_result.get(k), twap_result.get(k)) for k in key_metrics]

print("Run outputs (key fields):")
print_side_by_side_rows(rows)

print()
print("Full block_result:")
pprint(block_result)
print()
print("Full twap_result:")
pprint(twap_result)


Run outputs (key fields):
Metric                   | Block                    | TWAP                    
--------------------------------------------------------------------------------
strategy                 | Multi-Block              | Multi-TWAP (Horizon=20) 
net_requested_qty        | -11000                   | -11000                  
net_filled_qty           | -11000                   | -11000                  
arrival_vwap             | 28.13                    | 28.13                   
global_vwap              | 28.044712727272724       | 28.066361818181818      
slippage                 | 0.08528727272727465      | 0.06363818181818104     

Full block_result:
{'arrival_vwap': 28.13,
 'global_vwap': 28.044712727272724,
 'net_filled_qty': -11000,
 'net_requested_qty': -11000,
 'slippage': 0.08528727272727465,
 'strategy': 'Multi-Block'}

Full twap_result:
{'arrival_vwap': 28.13,
 'global_vwap': 28.066361818181818,
 'net_filled_qty': -11000,
 'net_requested_qty': -11000,
 'sli

In [9]:
# Section 5: Metrics and Slippage Comparison
# Compare the same metric set side-by-side, then compute simple differences.

# Fallback safety: if Section 4 was skipped, compute here.
if "block_result" not in globals() or "twap_result" not in globals():
    block_result = simulate_multi_order_block(parent_orders, ob_params, sim_params)
    twap_result = simulate_multi_order_twap(parent_orders, time_horizon, ob_params, sim_params)

metrics = ["net_requested_qty", "net_filled_qty", "arrival_vwap", "global_vwap", "slippage"]

rows = []
for m in metrics:
    b = block_result.get(m)
    t = twap_result.get(m)
    try:
        delta = float(b) - float(t)
        delta = round(delta, 6)
    except Exception:
        delta = None
    rows.append((m, b, t, delta))

print(f"{'Metric':<20} | {'Block':<16} | {'TWAP':<16} | {'Block - TWAP':<14}")
print("-" * 76)
for m, b, t, d in rows:
    print(f"{m:<20} | {str(b):<16} | {str(t):<16} | {str(d):<14}")

# Interpretation rule for this setup:
# Lower signed slippage is better (closer to arrival benchmark in favorable direction).
b_slip = block_result.get("slippage")
t_slip = twap_result.get("slippage")

print()
print("Interpretation:")
if b_slip is None or t_slip is None:
    print("- Could not compare slippage because one value is missing.")
elif abs(float(b_slip) - float(t_slip)) < 1e-12:
    print("- Block and TWAP are tied on slippage for this parameter set.")
elif float(b_slip) < float(t_slip):
    print("- Block has lower slippage in this run (better under this metric).")
else:
    print("- TWAP has lower slippage in this run (better under this metric).")

print("- Use this section together with Section 2 to explain why schedule shape impacts outcome.")


Metric               | Block            | TWAP             | Block - TWAP  
----------------------------------------------------------------------------
net_requested_qty    | -11000           | -11000           | 0.0           
net_filled_qty       | -11000           | -11000           | 0.0           
arrival_vwap         | 28.13            | 28.13            | 0.0           
global_vwap          | 28.044712727272724 | 28.066361818181818 | -0.021649     
slippage             | 0.08528727272727465 | 0.06363818181818104 | 0.021649      

Interpretation:
- TWAP has lower slippage in this run (better under this metric).
- Use this section together with Section 2 to explain why schedule shape impacts outcome.
